# درس ۰۲ - کاوش چارچوب Microsoft Agent

**چارچوب Microsoft Agent (MAF)** یک چارچوب یکپارچه برای ساخت عامل‌های هوش مصنوعی است. این چارچوب معماری تمیز و ترکیبی با چهار بلوک اصلی ارائه می‌دهد:

- **مشتری** – به نقطه پایان مدل هوش مصنوعی متصل می‌شود و ارتباط را مدیریت می‌کند
- **عامل** – مشتری را با دستورالعمل‌ها و تعاریف ابزارها می‌پوشاند
- **ابزارها** – قابلیت‌های عامل را با توابع سفارشی که مدل می‌تواند فراخوانی کند، گسترش می‌دهند
- **نشست** – تاریخچه گفتگو را برای تعاملات چندمرحله‌ای حفظ می‌کند

در این درس، ما یک **عامل رزرو سفر** می‌سازیم که با استفاده از این مفاهیم، در دسترس بودن مقصد را بررسی می‌کند.


## راه‌اندازی


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## درک معماری چارچوب Agent

چارچوب Agent مایکروسافت از معماری لایه‌ای پیروی می‌کند:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **کلاینت** – یک `FoundryChatClient` به استقرار Azure OpenAI متصل می‌شود. این کلاینت مسئول احراز هویت، قالب‌بندی درخواست‌ها و تجزیه پاسخ‌ها است.
2. **Agent** – که از طریق `provider.create_agent()` از کلاینت ایجاد می‌شود، این agent دسترسی به مدل را با دستورالعمل‌ها (prompt سیستم) و ابزارها ترکیب می‌کند.
3. **ابزارها** – توابع پایتون که با `@tool` تزئین شده‌اند و agent می‌تواند برای انجام عملیات یا بازیابی داده‌ها آن‌ها را فراخوانی کند.
4. **جلسه** – یک شیء `AgentSession` (ایجاد شده از طریق `agent.create_session()`) که تاریخچه مکالمه را ذخیره می‌کند و امکان گفتگوی چند مرحله‌ای را فراهم می‌سازد که agent زمینه قبلی را به خاطر می‌آورد.

بیایید هر لایه را گام به گام بسازیم.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## افزودن ابزارها با دکوراتور @tool

ابزارها به عامل‌ها اجازه می‌دهند تا فراتر از تولید متن عمل کنند. دکوراتور `@tool` یک تابع معمولی پایتون را به چیزی تبدیل می‌کند که عامل می‌تواند آن را فراخوانی کند.

نکات کلیدی:
- استفاده از `Annotated[type, "description"]` تا مدل هر پارامتر را درک کند.
- داک‌استرینگ تبدیل به توضیح ابزار می‌شود که مدل مشاهده می‌کند.
- `approval_mode="never_require"` به این معناست که ابزار به‌طور خودکار بدون تایید کاربر اجرا می‌شود.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ایجاد یک عامل با ابزارها

اکنون ما کلاینت، دستورالعمل‌ها و ابزارها را به یک عامل ترکیب می‌کنیم. `instructions` به عنوان پرامپت سیستم عمل می‌کنند — آن‌ها شخصیت و رفتار عامل را تعریف می‌کنند.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## مکالمات چند مرحله‌ای با جلسات

یک `AgentSession` (ایجاد شده از طریق `agent.create_session()`) تمام پیام‌های موجود در یک مکالمه را پیگیری می‌کند. با ارسال همان جلسه به هر فراخوانی `agent.run()`، عامل به کل سابقه مکالمه دسترسی دارد و می‌تواند به پیام‌های قبلی مراجعه کند.

ما `tools=[check_destination_availability]` را ارسال می‌کنیم تا عامل بتواند در هر مرحله بررسی‌کننده در دسترس بودن ما را فراخوانی کند.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## خلاصه

در این درس، شما چهار ستون اصلی چارچوب Microsoft Agent را بررسی کردید:

| مفهوم | آنچه یاد گرفتید |
|---------|------------------|
| **مشتری** | `FoundryChatClient` به Azure OpenAI با احراز هویت مبتنی بر اعتبارنامه متصل می‌شود |
| **عامل** | `provider.create_agent()` اتصال مدل را با دستورالعمل‌ها و نام بسته‌بندی می‌کند |
| **ابزارها** | دکوراتور `@tool` توابع پایتون را برای فراخوانی توسط عامل در دسترس قرار می‌دهد |
| **جلسه** | `agent.create_session()` تاریخچه گفتگو را در چند دور حفظ می‌کند |

این بلوک‌های ساختمانی با هم ترکیب می‌شوند تا عواملی بسازند که قادر به برقراری مکالمات طبیعی، فراخوانی توابع خارجی، و حفظ زمینه باشند — که پایه‌ای برای الگوهای پیشرفته‌تر عاملی در درس‌های بعدی است.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
